# 🚗 CoreVision — Car Brand+Model Classifier Training

**Model:** EfficientNetV2-S fine-tuned on VMMRdb (Vehicle Make & Model Recognition)  
**Dataset:** VMMRdb — uploaded to Google Drive as ZIP  
**Task:** Classify car **make + model + year** from web-scraped images  
**Runtime:** T4 GPU (~1-2 hours)

## Training Strategy: 2-Phase Progressive Unfreezing
- **Phase 1** (epochs 1-5): Backbone **frozen** → only head trains (fast)
- **Phase 2** (epochs 6-20): Backbone **unfrozen** → full fine-tuning

## Instructions
1. Upload `VMMRdb.zip` to Google Drive at `MyDrive/CoreVision/data/`
2. Go to **Runtime → Change runtime type → GPU (T4)**
3. Run all cells top to bottom
4. Final weights will be saved to your **Google Drive** at:
   `MyDrive/CoreVision/weights/car_classifier.pth`
5. Download and place in your project's `weights/` folder

In [ ]:
# ============================================================
# CELL 1: Check GPU & Mount Google Drive
# ============================================================
import torch

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/CoreVision/weights', exist_ok=True)
os.makedirs('/content/drive/MyDrive/CoreVision/data', exist_ok=True)
print('Drive mounted. Storage ready.')

In [ ]:
# ============================================================
# CELL 2: Install Dependencies
# ============================================================
!pip install -q timm
print('Dependencies installed!')

In [ ]:
# ============================================================
# CELL 3: Extract VMMRdb Dataset from Drive
# ============================================================
# Upload VMMRdb.zip to: MyDrive/CoreVision/data/VMMRdb.zip
# This cell extracts it and auto-detects the folder structure.
#
# VMMRdb structure is typically:
#   VMMRdb/
#     Make/
#       Model_Year/
#         image1.jpg
#         ...
#
# This cell finds the data root regardless of ZIP nesting.

import os
import zipfile
import time

LOCAL_DATA = '/content/data'
DATASET_DIR = f'{LOCAL_DATA}/vmmrdb'  # Final location after processing
os.makedirs(LOCAL_DATA, exist_ok=True)


def count_images(root):
    count = 0
    if not os.path.exists(root):
        return 0
    for _, _, files in os.walk(root):
        count += sum(1 for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')))
    return count


def find_data_root(search_dir):
    """Find the actual VMMRdb data root by looking for the level
    that has the most subdirectories containing images.
    
    VMMRdb can be:
      - Flat: root/Make_Model_Year/images  (1 level of class dirs)
      - Nested: root/Make/Model_Year/images (2 levels: make -> model)
    
    Returns (data_root, structure_type) where structure_type is
    'flat' or 'nested'.
    """
    best_root = search_dir
    best_count = 0
    best_type = 'flat'
    
    for root, dirs, files in os.walk(search_dir):
        # Skip too deep
        depth = root.replace(search_dir, '').count(os.sep)
        if depth > 3:
            continue
        
        # Count how many subdirectories here contain images directly
        img_subdirs = 0
        for d in dirs:
            dpath = os.path.join(root, d)
            has_imgs = any(f.lower().endswith(('.jpg', '.jpeg', '.png'))
                         for f in os.listdir(dpath) if os.path.isfile(os.path.join(dpath, f)))
            if has_imgs:
                img_subdirs += 1
        
        if img_subdirs > best_count:
            best_count = img_subdirs
            best_root = root
            best_type = 'flat'
        
        # Also check if this is a "make" level (dirs contain subdirs with images)
        nested_classes = 0
        for d in dirs:
            dpath = os.path.join(root, d)
            if not os.path.isdir(dpath):
                continue
            for sub in os.listdir(dpath):
                subpath = os.path.join(dpath, sub)
                if os.path.isdir(subpath):
                    has_imgs = any(f.lower().endswith(('.jpg', '.jpeg', '.png'))
                                 for f in os.listdir(subpath) if os.path.isfile(os.path.join(subpath, f)))
                    if has_imgs:
                        nested_classes += 1
        
        if nested_classes > best_count:
            best_count = nested_classes
            best_root = root
            best_type = 'nested'
    
    return best_root, best_type


# Check if already processed
if os.path.exists(DATASET_DIR) and count_images(DATASET_DIR) > 100:
    n = count_images(DATASET_DIR)
    class_dirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
    print(f'Dataset already ready: {n:,} images in {len(class_dirs)} class dirs')
else:
    # Find the ZIP file
    drive_data = '/content/drive/MyDrive/CoreVision/data'
    DRIVE_ZIP = None
    if os.path.exists(drive_data):
        for f in os.listdir(drive_data):
            if f.lower().endswith('.zip') and 'vmmr' in f.lower():
                DRIVE_ZIP = os.path.join(drive_data, f)
                break
    
    if DRIVE_ZIP is None:
        # Check for any ZIP
        for f in os.listdir(drive_data):
            if f.lower().endswith('.zip'):
                DRIVE_ZIP = os.path.join(drive_data, f)
                break
    
    if DRIVE_ZIP is None:
        print('Available files in Drive data dir:')
        !ls -lh "{drive_data}/"
        raise FileNotFoundError(f'No ZIP found in {drive_data}/')
    
    print(f'Found ZIP: {os.path.basename(DRIVE_ZIP)}')
    print(f'Extracting...')
    t0 = time.time()
    
    extract_dir = f'{LOCAL_DATA}/_vmmrdb_raw'
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(extract_dir)
    
    elapsed = time.time() - t0
    print(f'Extracted in {elapsed:.0f}s')
    
    # Show what was extracted
    print(f'\nExtracted contents:')
    for item in sorted(os.listdir(extract_dir))[:20]:
        item_path = os.path.join(extract_dir, item)
        if os.path.isdir(item_path):
            sub_count = len(os.listdir(item_path))
            print(f'  📁 {item}/ ({sub_count} items)')
        else:
            print(f'  📄 {item}')
    
    # Find the actual data root
    print(f'\nScanning for data root...')
    data_root, structure = find_data_root(extract_dir)
    print(f'Found data root: {data_root}')
    print(f'Structure type: {structure}')
    
    if structure == 'nested':
        # Flatten Make/Model_Year/ -> Make_Model_Year/
        print(f'\nFlattening nested structure (Make/Model_Year → Make_Model_Year)...')
        os.makedirs(DATASET_DIR, exist_ok=True)
        
        total_copied = 0
        total_classes = 0
        
        for make_name in sorted(os.listdir(data_root)):
            make_path = os.path.join(data_root, make_name)
            if not os.path.isdir(make_path):
                continue
            
            for model_name in sorted(os.listdir(make_path)):
                model_path = os.path.join(make_path, model_name)
                if not os.path.isdir(model_path):
                    continue
                
                # Create flat class name: Make_Model_Year
                class_name = f'{make_name}_{model_name}'
                class_dir = os.path.join(DATASET_DIR, class_name)
                os.makedirs(class_dir, exist_ok=True)
                
                images = [f for f in os.listdir(model_path)
                         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
                
                for img in images:
                    src = os.path.join(model_path, img)
                    dst = os.path.join(class_dir, img)
                    if not os.path.exists(dst):
                        os.symlink(src, dst)
                    total_copied += 1
                
                if images:
                    total_classes += 1
        
        print(f'Created {total_classes} class directories with {total_copied:,} images')
    
    else:  # flat structure
        # Just move/rename to DATASET_DIR
        if data_root != DATASET_DIR:
            if os.path.exists(DATASET_DIR):
                import shutil
                shutil.rmtree(DATASET_DIR)
            os.rename(data_root, DATASET_DIR)
            print(f'Moved {data_root} → {DATASET_DIR}')
    
    n = count_images(DATASET_DIR)
    class_dirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]
    print(f'\n✅ Dataset ready: {n:,} images in {len(class_dirs)} class directories')

# Show sample classes
class_dirs = sorted([d for d in os.listdir(DATASET_DIR)
                     if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f'\nSample classes: {class_dirs[:10]}')
print(f'Cell 3 complete.')

In [ ]:
# ============================================================
# CELL 4: Dataset Preparation (Year-Merge + Filter + Split)
# ============================================================
# Merges car years: e.g. Ford_Fiesta_2012 + Ford_Fiesta_2013 -> Ford_Fiesta
# Then filters by min samples and creates 80/20 train/val split.

import os
import re
import random
import shutil
from collections import defaultdict

VMMRDB_ROOT = '/content/data/vmmrdb'
SPLIT_ROOT = '/content/data/vmmrdb_split'
MIN_SAMPLES = 50         # Minimum images per merged class
VAL_RATIO = 0.2
MERGE_YEARS = True       # Merge model years into one class
FORCE_REBUILD = True


def is_image_file(name):
    return name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))


def strip_year(class_name):
    """Remove trailing year from class name.
    e.g. Acura_ILX_2013 -> Acura_ILX
         BMW_3_Series_2015 -> BMW_3_Series
    """
    return re.sub(r'[_ ](19|20)\d{2}$', '', class_name)


train_dir = f'{SPLIT_ROOT}/train'
val_dir = f'{SPLIT_ROOT}/val'

need_rebuild = FORCE_REBUILD or not os.path.exists(train_dir) or not os.path.exists(val_dir)

if need_rebuild:
    random.seed(42)

    # Clean up old split
    if os.path.exists(SPLIT_ROOT):
        shutil.rmtree(SPLIT_ROOT)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    # Scan all class directories
    all_dirs = sorted([
        d for d in os.listdir(VMMRDB_ROOT)
        if os.path.isdir(os.path.join(VMMRDB_ROOT, d))
        and not d.startswith('_')
    ])
    print(f'Found {len(all_dirs)} raw class directories')

    # Step 1: Merge years -> collect all images per merged class
    merged = defaultdict(list)  # merged_name -> [(src_path, filename), ...]

    for dir_name in all_dirs:
        dir_path = os.path.join(VMMRDB_ROOT, dir_name)
        if MERGE_YEARS:
            merged_name = strip_year(dir_name)
        else:
            merged_name = dir_name

        for f in os.listdir(dir_path):
            if is_image_file(f):
                src = os.path.join(dir_path, f)
                # Prefix filename with original dir to avoid collisions
                unique_name = f'{dir_name}__{f}'
                merged[merged_name].append((src, unique_name))

    print(f'After year-merge: {len(merged)} unique classes')

    # Step 2: Filter by min samples + split
    total_train = 0
    total_val = 0
    valid_classes = 0
    skipped = 0

    for cls_name in sorted(merged.keys()):
        images = merged[cls_name]

        if len(images) < MIN_SAMPLES:
            skipped += 1
            continue

        random.shuffle(images)
        split_idx = max(1, min(int(len(images) * (1 - VAL_RATIO)), len(images) - 1))

        train_imgs = images[:split_idx]
        val_imgs = images[split_idx:]

        os.makedirs(os.path.join(train_dir, cls_name), exist_ok=True)
        os.makedirs(os.path.join(val_dir, cls_name), exist_ok=True)

        for src, fname in train_imgs:
            dst = os.path.join(train_dir, cls_name, fname)
            shutil.copy2(src, dst)
        for src, fname in val_imgs:
            dst = os.path.join(val_dir, cls_name, fname)
            shutil.copy2(src, dst)

        total_train += len(train_imgs)
        total_val += len(val_imgs)
        valid_classes += 1

    print(f'\n Dataset preparation results:')
    print(f'   Total raw dirs:         {len(all_dirs)}')
    print(f'   After year-merge:       {len(merged)}')
    print(f'   Skipped (< {MIN_SAMPLES} imgs):  {skipped}')
    print(f'   Valid classes:          {valid_classes}')
    print(f'   Train images:           {total_train:,}')
    print(f'   Val images:             {total_val:,}')
    print(f'   Avg images/class:       {(total_train + total_val) / max(valid_classes, 1):.0f}')
else:
    print('Existing train/val split found. Reusing it.')

# Count classes
classes = sorted([c for c in os.listdir(train_dir)
                  if os.path.isdir(os.path.join(train_dir, c))])
n_classes = len(classes)
print(f'\nFinal dataset: {n_classes} classes')

# Save class names to Drive for inference
class_names_path = '/content/drive/MyDrive/CoreVision/data/vmmrdb_classes.txt'
with open(class_names_path, 'w') as f:
    f.write('\n'.join(classes))
print(f'Saved {n_classes} class names to Drive.')

In [ ]:
# ============================================================
# CELL 5: Hyperparameters & Configuration
# ============================================================
import os

SPLIT_ROOT = '/content/data/vmmrdb_split'

CONFIG = {
    # Data
    'train_dir': f'{SPLIT_ROOT}/train',
    'val_dir': f'{SPLIT_ROOT}/val',
    'num_classes': n_classes,
    'img_size': 224,

    # Training — shorter freeze, longer fine-tune
    'batch_size': 128,
    'num_epochs': 20,           # 5 frozen + 15 fine-tune
    'freeze_epochs': 5,         # Only 5 frozen epochs (head warmup)
    'num_workers': 4,

    # Optimizer
    'lr_backbone': 2e-4,       # More aggressive backbone LR for fine-tuning
    'lr_head': 1e-3,           # Head LR
    'weight_decay': 1e-4,

    # Scheduler
    'warmup_epochs': 2,

    # Checkpointing
    'save_path': '/content/drive/MyDrive/CoreVision/weights/car_classifier.pth',
    'resume_from': None,
}

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print()
print(f'Phase 1: Epochs 1-{CONFIG["freeze_epochs"]} — backbone FROZEN (head warmup)')
print(f'Phase 2: Epochs {CONFIG["freeze_epochs"]+1}-{CONFIG["num_epochs"]} — full fine-tune')

In [ ]:
# ============================================================
# CELL 6: Data Loaders with Augmentation
# ============================================================
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = CONFIG['img_size']

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1),
])

val_transforms = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.1)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(CONFIG['train_dir'], transform=train_transforms)
val_dataset = datasets.ImageFolder(CONFIG['val_dir'], transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=CONFIG['num_workers'],
    pin_memory=True, drop_last=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'],
    shuffle=False, num_workers=CONFIG['num_workers'],
    pin_memory=True,
    persistent_workers=True
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Classes: {len(train_dataset.classes)}')

In [ ]:
# ============================================================
# CELL 7: Build EfficientNetV2-S Model (with freeze support)
# ============================================================
import torch
import torch.nn as nn
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def build_model(num_classes, pretrained=True):
    backbone = timm.create_model(
        'tf_efficientnetv2_s',
        pretrained=pretrained,
        num_classes=0
    )
    feature_dim = backbone.num_features  # 1280
    model = nn.Sequential(
        backbone,
        nn.Dropout(0.3),
        nn.Linear(feature_dim, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(512, num_classes)
    )
    return model, feature_dim


def _get_raw(model):
    """Get the underlying nn.Sequential, unwrapping torch.compile if needed."""
    if hasattr(model, '_orig_mod'):
        return model._orig_mod
    return model


def freeze_backbone(model):
    raw = _get_raw(model)
    for param in raw[0].parameters():
        param.requires_grad = False
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'🧊 Backbone FROZEN — trainable params: {trainable:,}')


def unfreeze_backbone(model):
    raw = _get_raw(model)
    for param in raw[0].parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'🔥 Backbone UNFROZEN — trainable params: {trainable:,}')


def get_backbone_params(model):
    return list(_get_raw(model)[0].parameters())


def get_head_params(model):
    return list(_get_raw(model)[1:].parameters())


model, feature_dim = build_model(CONFIG['num_classes'])
model = model.to(device)

# Freeze backbone first, then compile
freeze_backbone(model)


total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params (Phase 1): {trainable:,}')
print(f'Feature dim: {feature_dim}')

In [ ]:
# ============================================================
# CELL 8: Optimizer, Scheduler & Loss
# ============================================================
import torch.optim as optim

head_params = [p for p in get_head_params(model) if p.requires_grad]

optimizer = optim.AdamW(
    head_params,
    lr=CONFIG['lr_head'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['freeze_epochs'],
    eta_min=1e-5
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler()

print('Phase 1 Optimizer: AdamW (head only)')
print(f'  Head LR: {CONFIG["lr_head"]}')
print('AMP: enabled')

In [ ]:
# ============================================================
# CELL 9: Training Loop (2-Phase Progressive Unfreezing)
# ============================================================
# ⚡ Colab-safe: saves best + latest checkpoint after EVERY epoch
# so training can resume if Colab disconnects.
import time

start_epoch = 0
best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

amp_enabled = device.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

SAVE_DIR = os.path.dirname(CONFIG['save_path'])
LATEST_PATH = os.path.join(SAVE_DIR, 'car_classifier_latest.pth')


def save_checkpoint(path, epoch, val_acc, val_top5=None, is_best=False):
    """Save checkpoint immediately to Drive."""
    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'accuracy': val_acc,
        'top5_accuracy': val_top5,
        'num_classes': CONFIG['num_classes'],
        'class_names': train_dataset.classes,
        'history': history,
        'best_val_acc': best_val_acc,
    }
    torch.save(ckpt, path)
    tag = '🏆 BEST' if is_best else '💾 latest'
    acc_str = f'{val_acc:.2%}'
    base = os.path.basename(path)
    print(f'  {tag} checkpoint saved → {base} (val_acc={acc_str})')


# Resume from checkpoint (try latest first, then best)
resume_path = CONFIG['resume_from']
if not resume_path or not os.path.exists(str(resume_path or '')):
    if os.path.exists(LATEST_PATH):
        resume_path = LATEST_PATH
        print('Found latest checkpoint to resume from!')
    elif os.path.exists(CONFIG['save_path']):
        resume_path = CONFIG['save_path']
        print('Found best checkpoint to resume from!')

if resume_path and os.path.exists(str(resume_path)):
    # Safety check: verify checkpoint class count matches current model
    _ckpt_pre = torch.load(resume_path, map_location='cpu')
    _ckpt_classes = _ckpt_pre.get('num_classes', None)
    if _ckpt_classes is not None and _ckpt_classes != CONFIG['num_classes']:
        print(f'⚠️ Checkpoint has {_ckpt_classes} classes but model has {CONFIG["num_classes"]}. Skipping resume.')
        resume_path = None
        del _ckpt_pre
    else:
        del _ckpt_pre

if resume_path and os.path.exists(str(resume_path)):
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt.get('best_val_acc', ckpt.get('accuracy', 0.0))
    history = ckpt.get('history', history)
    if start_epoch >= CONFIG['freeze_epochs']:
        unfreeze_backbone(model)
        optimizer = optim.AdamW([
            {'params': get_backbone_params(model), 'lr': CONFIG['lr_backbone']},
            {'params': get_head_params(model), 'lr': CONFIG['lr_head'] * 0.3},
        ], weight_decay=CONFIG['weight_decay'])
        ft_epochs = CONFIG['num_epochs'] - CONFIG['freeze_epochs']
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ft_epochs, eta_min=1e-6)
    best_str = f'{best_val_acc:.2%}'
    print(f'✅ Resumed from epoch {start_epoch}, best acc: {best_str}')


def train_epoch(model, loader, optimizer, criterion, scaler, device, epoch_idx, total_epochs):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    n_batches = len(loader)
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
        if batch_idx % 50 == 0 or batch_idx == n_batches - 1:
            pct = (batch_idx+1)/n_batches*100
            print(
                f'Epoch {epoch_idx+1}/{total_epochs} '
                f'| Batch {batch_idx+1}/{n_batches} ({pct:.0f}%) '
                f'| Loss: {loss.item():.4f}',
                end='\r'
            )
    return total_loss / total, total_correct / total


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, top5_correct, total = 0.0, 0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
            outputs = model(images)
            loss = criterion(outputs, labels)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        _, top5 = outputs.topk(min(5, outputs.size(1)), dim=1)
        top5_correct += top5.eq(labels.unsqueeze(1)).any(dim=1).sum().item()
        total_loss += loss.item() * images.size(0)
        total += images.size(0)
    return total_loss / total, total_correct / total, top5_correct / total


# ---- Main training loop ----
FREEZE_EPOCHS = CONFIG['freeze_epochs']
TOTAL_EPOCHS = CONFIG['num_epochs']
phase2_started = start_epoch >= FREEZE_EPOCHS

print(f'Starting training for {TOTAL_EPOCHS} epochs...')
print(f'  Phase 1 (epochs 1-{FREEZE_EPOCHS}): backbone frozen')
print(f'  Phase 2 (epochs {FREEZE_EPOCHS+1}-{TOTAL_EPOCHS}): full fine-tune')
print(f'  💾 Checkpoints saved to Drive after EVERY epoch (Colab-safe)')
print('=' * 60)

for epoch in range(start_epoch, TOTAL_EPOCHS):
    t0 = time.time()

    if epoch == FREEZE_EPOCHS and not phase2_started:
        phase2_started = True
        print('\n' + '=' * 60)
        print('🔥 PHASE 2: Unfreezing backbone for fine-tuning')
        print('=' * 60)
        unfreeze_backbone(model)
        optimizer = optim.AdamW([
            {'params': get_backbone_params(model), 'lr': CONFIG['lr_backbone']},
            {'params': get_head_params(model), 'lr': CONFIG['lr_head'] * 0.3},
        ], weight_decay=CONFIG['weight_decay'])
        ft_epochs = TOTAL_EPOCHS - FREEZE_EPOCHS
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ft_epochs, eta_min=1e-6)
        scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    phase_label = 'frozen' if epoch < FREEZE_EPOCHS else 'fine-tune'
    print(f'\nEpoch {epoch+1}/{TOTAL_EPOCHS} [{phase_label}]')

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, scaler, device,
        epoch_idx=epoch, total_epochs=TOTAL_EPOCHS
    )
    val_loss, val_acc, val_top5 = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    print(
        f'Epoch {epoch+1:02d}/{TOTAL_EPOCHS} [{phase_label}] '
        f'| Train Loss: {train_loss:.4f} Acc: {train_acc:.2%} '
        f'| Val Loss: {val_loss:.4f} Acc: {val_acc:.2%} Top-5: {val_top5:.2%} '
        f'| {elapsed:.0f}s'
    )

    # Save best model immediately
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(CONFIG['save_path'], epoch, val_acc, val_top5, is_best=True)

    # Always save latest checkpoint (for resume on Colab disconnect)
    save_checkpoint(LATEST_PATH, epoch, val_acc, val_top5, is_best=False)

print(f'\n🎉 Training complete! Best val accuracy: {best_val_acc:.2%}')
print(f'Best model: {CONFIG["save_path"]}')
print(f'Latest checkpoint: {LATEST_PATH}')


In [ ]:
# ============================================================
# CELL 10: Plot Training History
# ============================================================
import os
import matplotlib.pyplot as plt

if 'history' not in globals() or not isinstance(history, dict) or not history.get('train_loss'):
    loaded_history = None
    candidate_ckpts = []
    if 'CONFIG' in globals() and isinstance(CONFIG, dict):
        if CONFIG.get('resume_from'): candidate_ckpts.append(CONFIG['resume_from'])
        if CONFIG.get('save_path'): candidate_ckpts.append(CONFIG['save_path'])
    for ckpt_path in candidate_ckpts:
        if ckpt_path and os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu')
            maybe_history = ckpt.get('history')
            if isinstance(maybe_history, dict) and maybe_history.get('train_loss'):
                loaded_history = maybe_history
                print(f'Loaded history from: {ckpt_path}')
                break
    if loaded_history:
        history = loaded_history
    else:
        raise RuntimeError('No training history found. Run Cell 9 first.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)
freeze_ep = CONFIG.get('freeze_epochs', 5)

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=3)
axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val Loss', markersize=3)
axes[0].axvline(x=freeze_ep + 0.5, color='green', linestyle='--', alpha=0.7, label='Unfreeze')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=3)
axes[1].plot(epochs, [a*100 for a in history['val_acc']], 'r-o', label='Val Acc', markersize=3)
axes[1].axvline(x=freeze_ep + 0.5, color='green', linestyle='--', alpha=0.7, label='Unfreeze')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

plt.suptitle(f'EfficientNetV2-S on VMMRdb | Best Val: {best_val_acc:.2%}', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/CoreVision/training_history.png', dpi=150)
plt.show()
print('Plot saved to Drive!')

In [ ]:
# ============================================================
# CELL 11: Save Class Names & Download Weights
# ============================================================
from google.colab import files

with open('/content/vmmrdb_classes.txt', 'w') as f:
    f.write('\n'.join(train_dataset.classes))

print(f'Files in Google Drive at MyDrive/CoreVision/weights/:')
!ls -lh "/content/drive/MyDrive/CoreVision/weights/"
print()
print('Download the following files and put them in your project:')
print('  car_classifier.pth      -> project/weights/car_classifier.pth')
print('  vmmrdb_classes.txt      -> project/data/vmmrdb_classes.txt')

# files.download('/content/drive/MyDrive/CoreVision/weights/car_classifier.pth')
# files.download('/content/vmmrdb_classes.txt')